# 🚀 Mermaid Diagram Generator – Gemma-3-1B-IT + **Unsloth** (2× faster, 60% less VRAM)

**Optimized for Colab T4 (16 GB)**
- 4-bit NF4 quantization
- LoRA rank 16 targeting **all linear modules** (perfect for rigid Mermaid syntax)
- Unsloth 2× faster training + lower memory
- Full train → merge → INT8 .tflite for LiteRT edge deployment
- Custom Mermaid validation via `mmdc` (with sandbox fix)

Run cells sequentially. Training ≈ **45 min–1 hour** on T4.

In [ ]:
%%capture
!pip install -q --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-deps
!pip install -q unsloth_zoo
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes
!pip install -q ai-edge-torch-nightly

import torch
import ast
import subprocess
import os
import json
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

In [ ]:
# ===========================
# 2. Data Loading & Formatting
# ===========================
import re
import ast

# model_id = "google/gemma-3-1b-it"
model_id = "unsloth/gemma-3-1b-it-bnb-4bit" # 4-bit NF4 quantization

dataset = load_dataset("colinfrisch/diagrams-mermaid-filtered", split="train")
split_dataset = dataset.train_test_split(test_size=1000, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

def format_row(example):
    try:
        # The dataset already provides 'messages' as a list of dicts
        messages = example["messages"]

        # Ensure it is a list (handles cases where it might be a string in some versions)
        if isinstance(messages, str):
            messages = ast.literal_eval(messages)

        for msg in messages:
            if msg.get("role") == "assistant":
                msg["role"] = "model"

        formatted_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        return {"text": formatted_text}
    except Exception as e:
        return {"text": None}

# Initialize model and tokenizer (temporary for formatting only)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

train_dataset = train_dataset.map(format_row, remove_columns=train_dataset.column_names)
train_dataset = train_dataset.filter(lambda x: x["text"] is not None)

print(f"Training dataset size after filtering: {len(train_dataset)}")
if len(train_dataset) > 0:
    print("Example formatted text:")
    print(train_dataset[0]["text"][:500])
else:
    print("❌ Still empty. Check if tokenizer.apply_chat_template is failing.")

In [ ]:
# ===========================
# 3. Unsloth + QLoRA Setup (4-bit NF4 + all-linear)
# ===========================
max_seq_length = 2048
dtype = None  # Auto-detect (bfloat16 on T4)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    token=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()

In [ ]:
# ===========================
# 4. Training with SFTTrainer (GPU Optimized)
# ===========================
training_args = SFTConfig(
    output_dir="./gemma-mermaid-unsloth-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="adamw_8bit",
    logging_steps=10,
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=True,                             # Enable fp16 for T4 GPU
    bf16=False,                            # T4 does not support bf16
    max_seq_length=max_seq_length,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()
print("✅ Unsloth training completed on GPU")

In [ ]:
# ===========================
# 6. Merge LoRA → Standalone PyTorch (Unsloth fast merge)
# ===========================
model.save_pretrained_merged(
    "gemma-mermaid-merged",
    tokenizer,
    save_method="merged_16bit"   # full 16-bit weights, ready for edge conversion
)
print("✅ LoRA merged into standalone PyTorch model at ./gemma-mermaid-merged")